# Splink Prototype
Does Splink find repeat sales the exact key misses? Tested on Bristol (urban) and Powys (rural).

In [ ]:
import logging
import math

import duckdb
import pandas as pd
import splink.comparison_library as cl
from splink import DuckDBAPI, Linker, SettingsCreator, block_on

logging.getLogger("splink").setLevel(logging.ERROR)

SLICES = ["bristol", "powys"]
CLUSTER_THRESHOLD = 0.95

In [ ]:
def load(slice_name):
    # NULL = NULL is never true in blocking; fill so NULLs group like the exact key
    return duckdb.connect().sql(f"""
        SELECT * REPLACE (
            coalesce(postcode_clean, '<NONE>') AS postcode_clean,
            coalesce(street_clean,   '<NONE>') AS street_clean
        )
        FROM 'data/raw/{slice_name}_linkage_input.parquet'
    """).df()


# unit_key and paon_clean are hard rules (blocking only). property_type is left
# out: it is per-sale, not per-property, and disagrees within exact-key groups.
def make_settings():
    return SettingsCreator(
        link_type="dedupe_only",
        unique_id_column_name="transaction_id",
        blocking_rules_to_generate_predictions=[
            block_on("postcode_clean", "paon_clean", "unit_key"),
            block_on("street_clean", "paon_clean", "unit_key"),
        ],
        comparisons=[
            cl.ExactMatch("postcode_clean"),
            cl.JaroWinklerAtThresholds("street_clean", [0.9]),
        ],
        retain_intermediate_calculation_columns=True,
    )


def train(linker):
    linker.training.estimate_probability_two_random_records_match(
        [block_on("postcode_clean", "paon_clean", "street_clean", "unit_key")],
        recall=0.8,
    )
    linker.training.estimate_u_using_random_sampling(max_pairs=5e6, seed=1)
    # each pass can't estimate the column it blocks on: postcode from pass 2, street from pass 1
    linker.training.estimate_parameters_using_expectation_maximisation(
        block_on("postcode_clean", "paon_clean", "unit_key")
    )
    linker.training.estimate_parameters_using_expectation_maximisation(
        block_on("street_clean", "paon_clean", "unit_key")
    )


def street_weights(linker):
    """Match weights (bits) and u for the street comparison levels."""
    model = linker.misc.save_model_to_json()
    street = next(c for c in model["comparisons"] if c["output_column_name"] == "street_clean")
    out = {}
    for level in street["comparison_levels"]:
        if "m_probability" not in level:
            continue
        label = level["label_for_charts"]
        if label.startswith("Exact"):
            key = "exact"
        elif "Jaro-Winkler" in label:
            key = "jw"
        else:
            continue
        out[f"street_{key}_bits"] = round(math.log2(level["m_probability"] / level["u_probability"]), 2)
        if key == "exact":
            out["street_exact_u"] = level["u_probability"]
    return out


In [ ]:
GROUP_STATS = """
    WITH props AS (SELECT {col}, count(*) AS n_sales FROM joined GROUP BY 1)
    SELECT count(*)                             AS n_properties,
           count(*) FILTER (WHERE n_sales >= 2) AS n_with_resales,
           sum(n_sales) FILTER (WHERE n_sales >= 2) AS sales_in_repeat_groups,
           max(n_sales)                         AS biggest_group
    FROM props
"""

INVARIANTS = """
    SELECT
      (SELECT count(*) FROM (SELECT ek_id FROM joined GROUP BY 1
           HAVING count(DISTINCT cluster_id) > 1))      AS split_exact_keys,
      (SELECT count(*) FROM (SELECT cluster_id FROM joined GROUP BY 1
           HAVING count(DISTINCT ek_id) > 1))           AS clusters_merging_keys,
      (SELECT count(*) FROM (SELECT cluster_id FROM joined GROUP BY 1
           HAVING count(DISTINCT unit_key) > 1))        AS clusters_mixing_unit_key,
      (SELECT count(*) FROM (SELECT cluster_id FROM joined GROUP BY 1
           HAVING count(DISTINCT paon_clean) > 1))      AS clusters_mixing_paon
"""


def run_splink(slice_name):
    df = load(slice_name)
    linker = Linker(df, make_settings(), DuckDBAPI())
    train(linker)

    preds = linker.inference.predict(threshold_match_probability=0.01)
    clusters = linker.clustering.cluster_pairwise_predictions_at_threshold(
        preds, threshold_match_probability=CLUSTER_THRESHOLD
    )

    con = duckdb.connect()
    con.register("li", df)
    con.register("clusters_df", clusters.as_pandas_dataframe())
    con.register("preds_df", preds.as_pandas_dataframe())
    con.sql("""
        CREATE TABLE joined AS
        SELECT l.transaction_id, l.postcode_clean, l.paon_clean, l.street_clean,
               l.unit_key, c.cluster_id,
               dense_rank() OVER (
                   ORDER BY l.postcode_clean, l.paon_clean, l.street_clean, l.unit_key
               ) AS ek_id
        FROM li l JOIN clusters_df c USING (transaction_id)
    """)

    # pairs in different exact keys: all that fuzzy matching could add
    cross_key = con.sql("""
        SELECT p.match_probability, l.paon_clean, l.unit_key,
               l.postcode_clean AS pc_l, r.postcode_clean AS pc_r,
               l.street_clean AS st_l, r.street_clean AS st_r
        FROM preds_df p
        JOIN joined l ON p.transaction_id_l = l.transaction_id
        JOIN joined r ON p.transaction_id_r = r.transaction_id
        WHERE l.ek_id <> r.ek_id
        ORDER BY p.match_probability DESC, st_l
    """).df()

    # pairs sharing the exact key should score ~1
    within_key_median_p = con.sql("""
        SELECT median(p.match_probability)
        FROM preds_df p
        JOIN joined l ON p.transaction_id_l = l.transaction_id
        JOIN joined r ON p.transaction_id_r = r.transaction_id
        WHERE l.ek_id = r.ek_id
    """).fetchone()[0]

    stats = pd.concat(
        [con.sql(GROUP_STATS.format(col=col)).df().assign(method=method)
         for method, col in [("exact key", "ek_id"), ("splink", "cluster_id")]]
    )
    return {
        "stats": stats.assign(slice=slice_name),
        "invariants": con.sql(INVARIANTS).df().assign(slice=slice_name),
        "cross_key": cross_key,
        "street": street_weights(linker),
        "within_key_median_p": within_key_median_p,
    }

In [ ]:
results = {s: run_splink(s) for s in SLICES}

stats = pd.concat([r["stats"] for r in results.values()]).set_index(["slice", "method"])
display(stats)

In [ ]:
display(pd.concat([r["invariants"] for r in results.values()]).set_index("slice"))

In [ ]:
summary = pd.DataFrame({
    s: {
        "candidate_pairs_across_keys": len(r["cross_key"]),
        "best_match_probability": round(r["cross_key"].match_probability.max(), 3),
        "median_p_within_same_exact_key": round(r["within_key_median_p"], 3),
        "pairs_at_or_above_threshold": int((r["cross_key"].match_probability >= CLUSTER_THRESHOLD).sum()),
        **r["street"],
    }
    for s, r in results.items()
}).T
display(summary)

for s, r in results.items():
    print(s)
    display(r["cross_key"].head(10))

## Why Powys fails
25% of Powys rows have no street and carry the `'<NONE>'` sentinel, so unrelated rows "agree" on street 6.8% of the time (0.07% in Bristol). Exact street match drops to 3.86 bits (10.43), and a pair sharing the whole key scores ~0.46, never reaching 0.95.

## Verdict

| | Bristol | Powys |
|---|---|---|
| Exact key: properties / with resales | 112,803 / 60,772 | 30,910 / 14,070 |
| Splink: properties / with resales | 112,803 / 60,772 | 53,119 / 0 |
| Best cross-key match probability (threshold 0.95) | 0.072 | 0.477 |
| Merges, or clusters mixing unit / paon | 0 | 0 |

Splink matches the exact key on Bristol and loses every repeat sale on Powys. It never finds a repeat sale the exact key misses.

**Decision: retire Splink, use a deterministic key.**